In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import time

from data_processing import DataProcessor
from model import Model, TransferModel
from config import SlimConfig

In [2]:
K, LAM, ALPHA = 20, 100, 2.0  # model config you'd submit (see validate_loco.ipynb)

In [3]:
gene_names_list = pd.read_csv('../../data/vcc_2026_controls_final/gene_names.csv')['gene_name'].tolist()
dp = DataProcessor(gene_names_list)
kd = 0.9

In [4]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [5]:
train_names = list(DATASETS)                                   # all four training lines

model = TransferModel(gene_names_list, '../../data/embeddings/gene_string_embeddings.v0.3.h5', SlimConfig(), transfer_c = 50)
model.config.K, model.config.lambda_reg = K, LAM

measured_genes = {name: dp.load_dataset(name).var_names for name in train_names}
model.fit(*dp.build_Y(train_names, genes="union"), measured_genes)

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


In [6]:
pert_counts = pd.read_csv('../../data/vcc_2026_controls_final/pert_counts.csv')
target_genes = pert_counts['target_gene'].tolist()

In [7]:
source_controls = dp.source_ctrls(train_names)

In [8]:
context_D_data = sc.read_h5ad('../../data/vcc_2026_controls_final/context_D.h5ad')
context_E_data = sc.read_h5ad('../../data/vcc_2026_controls_final/context_E.h5ad')
context_F_data = sc.read_h5ad('../../data/vcc_2026_controls_final/context_F.h5ad')

context_dict = {'D': pd.Series(dp.context_ctrl_mean(context_D_data), index = gene_names_list),
                'E': pd.Series(dp.context_ctrl_mean(context_E_data), index = gene_names_list),
                'F': pd.Series(dp.context_ctrl_mean(context_F_data), index = gene_names_list)
                }

In [9]:
B_SCALE = 0.0
_orig_set_context = model.set_context
def _set_context_scaled(*args, **kwargs):
    _orig_set_context(*args, **kwargs)
    model.b_c = B_SCALE * model.b_c
model.set_context = _set_context_scaled

In [10]:
# check the weights for each context before writing anything
for ctx, ctrl in context_dict.items():
    model.set_context(ctrl, source_controls, tau=0.05, mix=0.3, panel=target_genes)
    print(ctx, dict(zip(model.source_names, model.weights.round(2))))

D {'K562': np.float64(0.36), 'RPE1': np.float64(0.16), 'HepG2': np.float64(0.2), 'Jurkat': np.float64(0.27)}
E {'K562': np.float64(0.1), 'RPE1': np.float64(0.65), 'HepG2': np.float64(0.16), 'Jurkat': np.float64(0.1)}
F {'K562': np.float64(0.13), 'RPE1': np.float64(0.58), 'HepG2': np.float64(0.16), 'Jurkat': np.float64(0.13)}


In [11]:
print("transfer_c:", model.transfer_c, "| max |b_c|:", np.abs(model.b_c).max())   # expect 50 and 0.0

transfer_c: 50 | max |b_c|: 0.0


In [12]:
# one entry per context, all keyed the same way; the validation set uses the same 300 targets everywhere
contexts_and_data = {'D': context_D_data, 'E': context_E_data, 'F': context_F_data}
target_genes_by_context = {c: target_genes for c in contexts_and_data}
panel_by_context = {c: target_genes for c in contexts_and_data}  # b_c is panel-matched to these targets

In [13]:
# streams one perturbation at a time (the full file built in memory would need ~33 GB);
# set_context runs inside write_submission, once per context, right before that context's cells
t = time.time()
model.write_submission(
    'FINAL_MODEL_slim_transfer_with_shrinkage.h5ad',
    contexts_and_data,                 # raw control cells to rescale (fully loaded, not backed)
    context_dict,                      # log1p control profiles -> similarity weights
    source_controls,
    panel_by_context,
    target_genes_by_context,
    tau=0.05,
    mix=0.3,
    knockdown_fraction=0.9,
    alpha=2.0,
    gain=2.5,                         # from fid_tuning.ipynb or fid_tuning_transfer.ipynb
    rounding="stochastic",
    cells_per_pert=400,                # required by the submission format
    rng=np.random.default_rng(0),      # reproducible cell sampling
)
print(f"written in {time.time() - t:.0f}s")

written in 302s


In [14]:
import anndata as ad

sub = ad.read_h5ad('FINAL_MODEL_slim_transfer_with_shrinkage.h5ad', backed='r')
counts = sub.obs.groupby(['context', 'target_gene'], observed=True).size()
print(sub.shape, "expected", (3 * len(target_genes) * 400, len(gene_names_list)))
print("cells per (context, target):", counts.unique(), "| groups:", len(counts))
print("gene order matches:", list(sub.var_names) == gene_names_list)
x = sub.X[:2000].data
print("integer, non-negative:", np.allclose(x, np.round(x)) and x.min() >= 0)

(360000, 18533) expected (360000, 18533)
cells per (context, target): [400] | groups: 900
gene order matches: True
integer, non-negative: True
